# Tạo bảng

In [18]:
import sqlite3
import pandas as pd

# 1. Tạo kết nối SQL
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Tạo bảng 'course'
cursor.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

# 3. Tạo bảng 'student'
cursor.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

# 4. Chèn dữ liệu vào bảng 'course'
courses = [(12, 'Giai tich'), (34, 'Thong ke'), (26, 'Tin hoc')]
cursor.executemany('INSERT INTO course VALUES (?, ?)', courses)

# 5. Chèn dữ liệu vào bảng 'student' (Các ô trống được để là None (NULL))
students = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?, ?, ?, ?, ?)', students)

print("Đã nhập xong dữ liệu và tạo bảng.")

# Hiển thị dữ liệu từ bảng 'course'
print("\nDữ liệu trong bảng 'course':")
cursor.execute('SELECT * FROM course')
courses_data = cursor.fetchall()
for course in courses_data:
    print(course)

# Hiển thị dữ liệu từ bảng 'student'
print("\nDữ liệu trong bảng 'student':")
cursor.execute('SELECT * FROM student')
students_data = cursor.fetchall()
for student in students_data:
    print(student)

Đã nhập xong dữ liệu và tạo bảng.

Dữ liệu trong bảng 'course':
(12, 'Giai tich')
(26, 'Tin hoc')
(34, 'Thong ke')

Dữ liệu trong bảng 'student':
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7)
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2)
(3, 'Pham Van Nam', 'Toan Tin', None, 7.9)
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2)
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0)
(6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5)
(7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2)
(8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8)
(9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2)
(10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)


Kết luận:
- Bảng student có các mã môn học 20 và 24, nhưng các mã này không tồn tại trong bảng course.
- Ngược lại, bảng course có môn "Tin hoc" mã 26, nhưng không có sinh viên nào trong bảng student đăng ký mã này.
- Có 3 sinh viên (ID: 3, 9, 10) hoàn toàn chưa có mã môn học.

# Phần 1. Các phép kết nối

Tích Decartes

In [20]:
query_cartesian = "SELECT * FROM student CROSS JOIN course"
df_cartesian = pd.read_sql(query_cartesian, conn)
# Hiển thị tất cả
display(df_cartesian)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


INNER JOIN

In [ ]:
query_inner = """
    SELECT s.*, c.course_name 
    FROM student s
    INNER JOIN course c ON s.course_id = c.id
"""
df_inner = pd.read_sql(query_inner, conn)
display(df_inner)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


Kết luận: Làm mất mát rất nhiều dữ liệu. Chỉ còn lại 3 dòng (Sinh viên 1, 2, 7) vì đây là những người duy nhất có mã môn học hợp lệ và tồn tại ở cả hai phía.

LEFT JOIN

In [23]:
query_left = """
    SELECT s.*, c.course_name 
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
"""
df_left = pd.read_sql(query_left, conn)
display(df_left)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


Kết luận: Đây là cách tốt nhất nếu muốn giữ lại danh sách đầy đủ 10 sinh viên để theo dõi điểm số, dù một số người chưa có tên môn học cụ thể.

RIGHT JOIN

In [24]:
df_student = pd.read_sql("SELECT * FROM student", conn)
df_course = pd.read_sql("SELECT * FROM course", conn)

df_right = pd.merge(df_student, df_course, left_on='course_id', right_on='id', how='right')

print("RIGHT JOIN:")
display(df_right)

RIGHT JOIN:


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,NaN,NaN,NaN,NaN,26,Tin hoc
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
3,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke


Kết luận: Kết quả cho thấy môn "Tin hoc" hiện đang trống (Không có người học), đồng thời loại bỏ các sinh viên có mã môn 20, 24 hoặc không có mã môn.

FULL OUTER JOIN

In [25]:
df_full = pd.merge(df_student, df_course, left_on='course_id', right_on='id', how='outer')
print("\nFULL OUTER JOIN:")
display(df_full)


FULL OUTER JOIN:


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,NaN
2,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,NaN
3,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,NaN
4,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,26.0,Tin hoc
6,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,NaN
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,NaN


Kết luận: Kết quả chỉ ra cả những sinh viên có mã môn không tồn tại và những môn học không có sinh viên.

# Phần 2

Cập nhật và Làm sạch dữ liệu

In [27]:
# 1. Cập nhật các giá trị course_id còn thiếu (NULL)
# Gán sinh viên 3 học môn 12, sinh viên 9 học môn 34, sinh viên 10 học môn 26
cursor.execute("UPDATE student SET course_id = 12 WHERE student_id = 3")
cursor.execute("UPDATE student SET course_id = 34 WHERE student_id = 9")
cursor.execute("UPDATE student SET course_id = 26 WHERE student_id = 10")

# 2. Loại bỏ những bản ghi học môn không tồn tại trong bảng course (mã 20, 24)
cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course)")

conn.commit()

print("Đã cập nhật và làm sạch dữ liệu.")
# Hiển thị bảng student sau khi làm sạch
df_cleaned = pd.read_sql("SELECT * FROM student", conn)
display(df_cleaned)

Đã cập nhật và làm sạch dữ liệu.


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,12,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,34,7.2
5,10,Cao Thi Hanh,May Tinh,26,7.0


Kết luận: Danh sách sinh viên bị rút gọn đáng kể (Từ 10 xuống còn 6 sinh viên).
- Các sinh viên bị loại là: Lê Thanh Huyền, Vũ Quốc Anh, Đặng Thùy Linh, Hổ Ngọc Mai do họ đăng ký các mã môn học (20, 24) không có trong danh mục chính thức.
- Việc cập nhật giúp các môn học như "Tin học" (mã 26) vốn không có ai học nay đã có sinh viên tham gia (sinh viên Cao Thị Hạnh).

Thực hiện các yêu cầu thống kê

a. Tổng số sinh viên, điểm trung bình của từng lớp

In [28]:
query_a = """
    SELECT 
        class AS [Lớp], 
        COUNT(student_id) AS [Tổng số SV], 
        ROUND(AVG(score), 2) AS [Điểm TB Lớp]
    FROM student
    GROUP BY class
"""
df_a = pd.read_sql(query_a, conn)
print("a. Thống kê theo Lớp:")
display(df_a)

a. Thống kê theo Lớp:


,Lớp,Tổng số SV,Điểm TB Lớp
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


Kết luận:
- Lớp Kinh Tế đang dẫn đầu về số lượng sinh viên (3 người) và có điểm trung bình khá cao (~8.53).
- Lớp Toán Tin chỉ còn lại 1 sinh viên sau khi lọc dữ liệu.

b + c. Thống kê theo môn học và xếp loại thi đua

In [30]:
query_bc = """
SELECT 
    c.course_name AS [Tên Môn Học],
    COUNT(s.student_id) AS [Tổng số SV],
    ROUND(AVG(s.score), 2) AS [Điểm TB Môn],
    CASE 
        WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
        WHEN AVG(s.score) >= 6.0 THEN 'Tốt'
        ELSE 'Kém'
    END AS [Phân loại]
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
df_bc = pd.read_sql(query_bc, conn)
print("b & c. Thống kê theo môn học và xếp loại:")
display(df_bc)

b & c. Thống kê theo môn học và xếp loại:


,Tên Môn Học,Tổng số SV,Điểm TB Môn,Phân loại
0,Giai tich,2,7.30,Tốt
1,Thong ke,3,8.53,Tốt
2,Tin hoc,1,7.00,Tốt


Kết luận:
- Môn Thống kê có số lượng sinh viên đông nhất và đạt xếp loại Tốt (8.53).
- Môn Giải tích có điểm trung bình 7.3, cũng xếp loại Tốt.
- Môn Tin học (Sau khi gán sinh viên) có điểm trung bình là 7.0, xếp loại Tốt.
- Trong bộ dữ liệu sau khi lọc, hiện tại không có môn nào đạt mức Xuất sắc (Vì không môn nào có điểm trung bình đạt 9.0) hoặc Kém (Tất cả đều trên 6.0).

# Phần 3. Xếp hạng sinh viên

a. Xếp hạng theo điểm số

In [31]:
query_overall = """
SELECT *,
    RANK() OVER (ORDER BY score DESC) as Rank_High,
    RANK() OVER (ORDER BY score ASC) as Rank_Low
FROM student
"""
df_overall = pd.read_sql(query_overall, conn)

print("--- XẾP HẠNG TOÀN TRƯỜNG ---")
print("Top 3 điểm cao nhất:")
display(df_overall[df_overall['Rank_High'] <= 3].sort_values('Rank_High'))

print("\nTop 3 điểm thấp nhất:")
display(df_overall[df_overall['Rank_Low'] <= 3].sort_values('Rank_Low'))

--- XẾP HẠNG TOÀN TRƯỜNG ---
Top 3 điểm cao nhất:


,student_id,name,class,course_id,score,Rank_High,Rank_Low
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,5
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,5
2,3,Pham Van Nam,Toan Tin,12,7.9,3,4



Top 3 điểm thấp nhất:


,student_id,name,class,course_id,score,Rank_High,Rank_Low
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6,1
4,10,Cao Thi Hanh,May Tinh,26,7.0,5,2
3,9,Duong Huu Phuc,Kinh Te,34,7.2,4,3


Kết luận: 
- Top 3 cao nhất: Đồng hạng nhất là Tran Thi Lan và Bui Tien Dung (9.2 điểm). Đứng thứ ba là Pham Van Nam (7.9 điểm).
- Top 3 thấp nhất: Thấp nhất là Nguyen Minh Hoang (6.7 điểm), tiếp theo là Cao Thi Hanh (7.0 điểm) và Duong Huu Phuc (7.2 điểm).

b. Xếp hạng theo lớp học

In [37]:
query_class = """
SELECT *,
    RANK() OVER (PARTITION BY class ORDER BY score DESC) as Rank_High,
    RANK() OVER (PARTITION BY class ORDER BY score ASC) as Rank_Low
FROM student
"""
df_class = pd.read_sql(query_class, conn)

print("--- XẾP HẠNG THEO LỚP ---")
# Lọc Top 3 cho mỗi lớp (vì danh sách ít nên có thể hiện tất cả)
display(df_class[(df_class['Rank_High'] <= 3) | (df_class['Rank_Low'] <= 3)])

--- XẾP HẠNG THEO LỚP ---


,student_id,name,class,course_id,score,Rank_High,Rank_Low
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,2
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3,1
3,10,Cao Thi Hanh,May Tinh,26,7.0,1,2
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2,1
5,3,Pham Van Nam,Toan Tin,12,7.9,1,1


Kết luận: 
- Lớp Kinh Tế: Có sự cạnh tranh gắt gao nhất khi cả 2 sinh viên đứng đầu toàn trường đều nằm ở lớp này. Duong Huu Phuc tuy có điểm khá (7.2 điểm) nhưng lại đứng cuối lớp Kinh Tế.
- Lớp Máy Tính: Chỉ còn 2 sinh viên, Cao Thi Hanh (7.0 điểm) xếp trên Nguyen Minh Hoang (6.7 điểm).
- Lớp Toán Tin: Chỉ còn duy nhất Pham Van Nam là đứng cuối lớp này.

c. Xếp hạng theo mã môn học

In [36]:
query_course = """
SELECT s.*, c.course_name,
    RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) as Rank_High,
    RANK() OVER (PARTITION BY s.course_id ORDER BY s.score ASC) as Rank_Low
FROM student s
JOIN course c ON s.course_id = c.id
"""
df_course = pd.read_sql(query_course, conn)

print("--- XẾP HẠNG THEO MÔN HỌC ---")
display(df_course[(df_course['Rank_High'] <= 3) | (df_course['Rank_Low'] <= 3)])

--- XẾP HẠNG THEO MÔN HỌC ---


,student_id,name,class,course_id,score,course_name,Rank_High,Rank_Low
0,3,Pham Van Nam,Toan Tin,12,7.9,Giai tich,1,2
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich,2,1
2,10,Cao Thi Hanh,May Tinh,26,7.0,Tin hoc,1,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke,1,2
4,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke,1,2
5,9,Duong Huu Phuc,Kinh Te,34,7.2,Thong ke,3,1


Kết luận:
- Môn Thống kê (34): Đây là môn có điểm số cao nhất với hai điểm 9.2.
- Môn Giải tích (12): Pham Van Nam (7.9) xếp trên Nguyen Minh Hoang (6.7).
- Môn Tin học (26): Chỉ có 1 sinh viên duy nhất là Cao Thi Hanh (7.0).

# Phần 4

In [38]:
# 1. Thêm trường graduation_date vào bảng student
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
    conn.commit()
    print("Đã thêm cột graduation_date thành công.")
except sqlite3.OperationalError:
    print("Cột graduation_date đã tồn tại.")

# 2. Cập nhật ngày tốt nghiệp: Ngày hiện tại + số hạng (theo điểm số DESC)
# Chúng ta sử dụng Common Table Expression (CTE) để tính hạng trước khi update
query_update_date = """
WITH StudentRanks AS (
    SELECT 
        student_id, 
        RANK() OVER (ORDER BY score DESC) as rank_val
    FROM student
)
UPDATE student
SET graduation_date = datetime('now', '+' || (
    SELECT rank_val FROM StudentRanks WHERE StudentRanks.student_id = student.student_id
) || ' days')
"""

cursor.execute(query_update_date)
conn.commit()

# Hiển thị kết quả
df_final = pd.read_sql("SELECT student_id, name, score, graduation_date FROM student ORDER BY score DESC", conn)
display(df_final)

Đã thêm cột graduation_date thành công.


,student_id,name,score,graduation_date
0,2,Tran Thi Lan,9.2,2026-04-04 14:06:36
1,7,Bui Tien Dung,9.2,2026-04-04 14:06:36
2,3,Pham Van Nam,7.9,2026-04-06 14:06:36
3,9,Duong Huu Phuc,7.2,2026-04-07 14:06:36
4,10,Cao Thi Hanh,7.0,2026-04-08 14:06:36
5,1,Nguyen Minh Hoang,6.7,2026-04-09 14:06:36


Kết luận:

1. Tính logic của dữ liệu:
- Cột graduation_date đã được điền tự động dựa trên thứ hạng điểm số của sinh viên.
- Theo công thức $Thời\_gian\_hiện\_tại + Thứ\_hạng$, những sinh viên có thứ hạng cao nhất (Hạng 1) sẽ có ngày tốt nghiệp gần với hiện tại nhất (Hiện tại + 1 ngày).
- Những sinh viên có thứ hạng thấp hơn sẽ có ngày tốt nghiệp xa hơn trong tương lai.

2. Chi tiết kết quả:
- Trần Thị Lan và Bùi Tiến Dũng (Đồng hạng 1) có cùng ngày tốt nghiệp (Ví dụ: Nếu hôm nay là 03/04/2026, ngày tốt nghiệp sẽ là 2026-04-04).
- Nguyễn Minh Hoàng (Hạng 6 - Thấp nhất sau khi lọc) sẽ có ngày tốt nghiệp xa nhất (Hiện tại + 6 ngày, tức là 2026-04-09).

3. Về mặt kỹ thuật SQL:
- Việc sử dụng datetime('now', '+N days') là cách xử lý thời gian linh hoạt trong SQLite để thực hiện các phép toán cộng trừ ngày tháng.
- Việc kết hợp WITH ... AS (CTE) giúp câu lệnh UPDATE trở nên rõ ràng hơn khi cần lấy dữ liệu từ một bảng ảo có chứa hàm cửa sổ (RANK()).